In [12]:
import pandas as pd
import numpy as np


df = pd.read_csv('data/bank+marketing/bank-additional/bank-additional-full.csv', sep=';')

total_visitors = len(df)

qualified_leads = len(df[df['campaign']<= 3])

conversions = len(df[df['y']== 'yes'])

cr_global = (conversions / total_visitors)*100

cr_lead_to_customer = (conversions / qualified_leads)*100

drop_off_stage1 = ((total_visitors-qualified_leads)/total_visitors)*100

drop_off_total = ((total_visitors-conversions)/total_visitors)*100

funnel_summary = pd.DataFrame({
    'Étape du Funnel': ['1. Outreach Total (Prospects)', '2. Leads Engagés (<=3 contacts)', '3. Clients Convertis (Souscriptions)'],
    'Volume': [total_visitors, qualified_leads, conversions],
    '% du Trafic Total': [100.0, (qualified_leads / total_visitors) * 100, cr_global],
    'Taux de Rétention Étape (%)': [100.0, 100.0 - drop_off_stage1, cr_lead_to_customer]
    
})
print("=== APERÇU DU FUNNEL GLOBAL ===")
print(funnel_summary.to_string(index=False))
print(f"\n- Taux de conversion global : {cr_global:.2f}%")
print(f"- Perte de prospects par sur-sollicitation (>3 appels) : {drop_off_stage1:.2f}%")

=== APERÇU DU FUNNEL GLOBAL ===
                     Étape du Funnel  Volume  % du Trafic Total  Taux de Rétention Étape (%)
       1. Outreach Total (Prospects)   41188         100.000000                   100.000000
     2. Leads Engagés (<=3 contacts)   33553          81.463047                    81.463047
3. Clients Convertis (Souscriptions)    4640          11.265417                    13.828868

- Taux de conversion global : 11.27%
- Perte de prospects par sur-sollicitation (>3 appels) : 18.54%


In [18]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/bank+marketing/bank-additional/bank-additional-full.csv', sep=';')

channel_analysis = df.groupby('contact').agg(
    Prospects=('y', 'count'),
    Conversions=('y', lambda x: round((x=='yes').mean() * 100, 2))
).reset_index()


print("=== 1. PERFORMANCE PAR CANAL DE CONTACT ===")
print(channel_analysis.to_string(index=False))

=== 1. PERFORMANCE PAR CANAL DE CONTACT ===
  contact  Prospects  Conversions
 cellular      26144        14.74
telephone      15044         5.23


In [15]:
campaign_fatigue = df.groupby('campaign').agg(
    Volume_Prospects = ('y', 'count'),
    Conversions=('y', lambda x:(x == 'yes').sum()),
    Taux_Conversion = ('y', lambda x: round((x == 'yes').mean()*100, 2))
).reset_index().head(6)

print("\n=== 2. IMPACT DE LA SUR-SOLLICITATION (FATIGUE LEAD) ===")
print(campaign_fatigue.to_string(index=False))




=== 2. IMPACT DE LA SUR-SOLLICITATION (FATIGUE LEAD) ===
 campaign  Volume_Prospects  Conversions  Taux_Conversion
        1             17642         2300            13.04
        2             10570         1211            11.46
        3              5341          574            10.75
        4              2651          249             9.39
        5              1599          120             7.50
        6               979           75             7.66


In [16]:
job_analysis = df.groupby('job').agg(
    Total=('y', 'count'),
    Conversions=('y', lambda x: round(x=='yes').sum()),
    Taux_Conversion = ('y', lambda x: round((x == 'yes').mean()*100, 2))
).sort_values(by='Taux_Conversion', ascending=False).reset_index()
print("\n=== 3. CONVERSION PAR SEGMENT PROMOTIONNEL / MÉTIER ===")
print(job_analysis.to_string(index=False))


=== 3. CONVERSION PAR SEGMENT PROMOTIONNEL / MÉTIER ===
          job  Total  Conversions  Taux_Conversion
      student    875          275            31.43
      retired   1720          434            25.23
   unemployed   1014          144            14.20
       admin.  10422         1352            12.97
   management   2924          328            11.22
      unknown    330           37            11.21
   technician   6743          730            10.83
self-employed   1421          149            10.49
    housemaid   1060          106            10.00
 entrepreneur   1456          124             8.52
     services   3969          323             8.14
  blue-collar   9254          638             6.89


In [19]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/bank+marketing/bank-additional/bank-additional-full.csv', sep=';')

poutcome_analysis = df.groupby('poutcome').agg(
    
    Volume_Prospects=('y', 'count'),
    Conversions=('y', lambda x: (x == 'yes').sum()),
    Taux_Conversion = ('y', lambda x: round((x == 'yes').mean()*100,2))
).reset_index().sort_values(by='Taux_Conversion', ascending=False)

print("=== 1. IMPACT DU PASSÉ PROSPECT (POUTCOME) ===")
print(poutcome_analysis.to_string(index=False))

=== 1. IMPACT DU PASSÉ PROSPECT (POUTCOME) ===
   poutcome  Volume_Prospects  Conversions  Taux_Conversion
    success              1373          894            65.11
    failure              4252          605            14.23
nonexistent             35563         3141             8.83


In [20]:
import pandas as pd
import numpy as np

df['deja_contacte'] = df['pdays'].apply(lambda x: 'Jamais Contactés (999)' if x == 999 else 'Deja Contactés (<999)')

pdays_summary = df.groupby('deja_contacte').agg(
    Volume=('y','count'),
    Conversions= ('y', lambda x: (x == 'yes').sum()),
    Taux_Conversion=('y', lambda x: round((x == 'yes').mean()*100,2))
      
).reset_index()

print("\n=== 2. EFFET DES RELANCES DE BASE PROSPECT (PDAYS) ===")
print(pdays_summary.to_string(index=False))


=== 2. EFFET DES RELANCES DE BASE PROSPECT (PDAYS) ===
         deja_contacte  Volume  Conversions  Taux_Conversion
 Deja Contactés (<999)    1515          967            63.83
Jamais Contactés (999)   39673         3673             9.26


In [22]:
month_order = ['mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
month_analysis = df.groupby('month').agg(
    Volume_Contacts=('y', 'count'),
    Conversions=('y', lambda x: (x == 'yes').sum()),
    Taux_Conversion=('y', lambda x: round((x == 'yes').mean() * 100, 2))
).reindex(month_order).dropna().reset_index()

print("\n=== 3. SAISONNALITÉ MENSUELLE DE LA CONVERSION ===")
print(month_analysis.to_string(index=False))


=== 3. SAISONNALITÉ MENSUELLE DE LA CONVERSION ===
month  Volume_Contacts  Conversions  Taux_Conversion
  mar              546          276            50.55
  apr             2632          539            20.48
  may            13769          886             6.43
  jun             5318          559            10.51
  jul             7174          649             9.05
  aug             6178          655            10.60
  sep              570          256            44.91
  oct              718          315            43.87
  nov             4101          416            10.14
  dec              182           89            48.90


In [23]:
import pandas as pd

# Simulation du gain d'efficacité en supprimant les appels inefficaces (>3 contacts)
appels_totaux = len(df)
appels_efficaces = len(df[df['campaign'] <= 3])
appels_gaspilles = appels_totaux - appels_efficaces

conversions_totales = (df['y'] == 'yes').sum()
conversions_efficaces = (df[df['campaign'] <= 3]['y'] == 'yes').sum()

print("=== IMPACT FINANCIER & CRO ===")
print(f"- Volume total d'appels passés : {appels_totaux:,}")
print(f"- Appels économisables (>3 contacts) : {appels_gaspilles:,} ({(appels_gaspilles/appels_totaux)*100:.1f}% de l'effort commercial)")
print(f"- Conversions préservées avec la règle des 3 contacts : {(conversions_efficaces/conversions_totales)*100:.1f}%")

=== IMPACT FINANCIER & CRO ===
- Volume total d'appels passés : 41,188
- Appels économisables (>3 contacts) : 7,635 (18.5% de l'effort commercial)
- Conversions préservées avec la règle des 3 contacts : 88.0%


In [24]:
df.to_excel('data/bank+marketing/bank-additional/bank-additional-full.xlsx', index=False)